#Evaluation Matrix

Install Libraries

In [ ]:
%pip install scikit-learn

In [ ]:
%pip install pandas

In [ ]:
%pip install numpy

In [ ]:
%pip install matplotlib

In [ ]:
%pip install seaborn

Library Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn datasets & model selection
from sklearn.datasets import(make_classification, make_regression,
                             load_breast_cancer, load_diabetes)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

#Regression Model
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

#Evaluation matrics - THE CORE OF THIS NOTEBOOK
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
    ConfusionMatrixDisplay
)

Plotting Style

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print(" All libraries loaded successfully")
print(f"Numpy: {np.__version__} | Pandas: {pd.__version__}")

Dataset Preparation

In [ ]:
#Classification Breast Cancer Dataset
cancer = load_breast_cancer()
X_clf, y_clf = cancer.data, cancer.target   #0=Malignant, 1=benign
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

#Feature scaling - important for Logistic Regression and SVM
scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(X_train_c)
X_test_c = scaler_c.fit_transform(X_test_c)

#Diabaetes Dataset
diabetes = load_diabetes()
X_reg, y_reg = diabetes.data, diabetes.target
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
scaler_r = StandardScaler()
X_train_r = scaler_r.fit_transform(X_train_r)
X_test_r = scaler_r.fit_transform(X_test_r)

#Quick Data Summery
print("CLASSIFICATION DATASET (Breast Cancer)")
print(f"   Train: {X_train_c.shape}  |  Test: {X_test_c.shape}")
print(f"   Class balance - Malignant: {(y_clf==0).sum()}  |  Benign: {(y_clf==1).sum()}\n")
print("REGRESSION DATASET (Diabetes)")
print(f"   Train: {X_train_r.shape}  |  Test: {X_test_r.shape}")
print(f"   Target range: [{y_reg.min():.0f}, {y_reg.max():.0f}]")

Student Task

In [ ]:
%matplotlib inline

# 1. Print the first 5 feature names of breast cancer dataset
print("Feature names: ", cancer.feature_names[:5])

#2. Calculate and print Imbalence ratio
Malignant = (y_clf == 0).sum()
Benign = (y_clf == 1).sum()
Imbalence_Ratio = Malignant/Benign
print(f"Imbalance Ratio: {Imbalence_Ratio:.2f}")

#3. Plot a bar chart showing class distribution
plt.figure(figsize=(6, 4))
classes = ['Malignant', 'Benign']
count = [Malignant, Benign]
bars = plt.bar(classes, count, color = ["tab:blue", "tab:orange"])
for bar, val in zip(bars, count):
  plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{val}", ha='center', va='bottom', fontsize = 11, fontweight = 'bold')
plt.title("Class Distribution -- Breast Cancer Dataset")
plt.show()

Accuracy and Confusion Matrix

In [ ]:
#Train three classifier for comparision

models = {
    'Logistic Regression': LogisticRegression(max_iter = 1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradiant Boosting': GradientBoostingClassifier(n_estimators=100, random_state = 42)
}

results = {}

for name, model in models.items():
  model.fit(X_train_c, y_train_c)
  y_pred = model.predict(X_test_c)
  y_proba = model.predict_proba(X_test_c)[:, 1]

  results[name] = {
      'model': model,
      'y_pred': y_pred,
      'y_proba': y_proba,
      'accuracy': accuracy_score(y_test_c, y_pred),
      'cm': confusion_matrix(y_test_c, y_pred)
  }

  print(f"{name}: Accuracy = {results[name]['accuracy']:.4f}")

  # ----- Visualise confusion matrices side by side --------

  fig, axes = plt.subplots(1, 3, figsize=(15, 4))

  for ax, (name, res) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(
        confusion_matrix = res['cm'],
        display_labels=cancer.target_names
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{name}\nAcc: {res['accuracy']:.3f}", fontsize=11)
  plt.tight_layout()
  plt.suptitle("Confusion Matrices - All Models", y=1.02, fontsize=13, fontweight='bold')
  plt.show()

Student Work

In [ ]:
#1. Pick the Logistic Regression model's confusion matrix
Log_Reg_CN = results['Logistic Regression']['cm']
print(Log_Reg_CN)

#2. Extract TP, FP, TN, FN
TP = Log_Reg_CN[1][1]
TN = Log_Reg_CN[0][0]
FP = Log_Reg_CN[0][1]
FN = Log_Reg_CN[1][0]
print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")

#3. Calculate accuracy from scratch (without sklearn) and verify it matches
manual_acc = (TP+TN)/(TP+TN+FP+FN)
print(f"Manual Accuracy: {manual_acc:.4f}")
sklearn_acc = results['Logistic Regression']['accuracy']
print(f"Sklearn Accuracy: {sklearn_acc:.4f}")